## Тема: алгоритмическая и пространственная сложность

Два вопроса, которые можно задать про любой код:

1. Насколько он замедлится, если данных станет больше?
2. Сколько лишней памяти он при этом займёт?

Пойдём так: сначала руками замерим время и память, увидим закономерности, и только потом введём для них короткую запись - O-нотацию.

# 1. Замеряем время

## 1.1. Как замерить время работы кода

`time.perf_counter()` возвращает число секунд, прошедших с какого-то произвольного момента. Само это число бесполезно, а вот разность двух вызовов - это время работы кода между ними.

In [1]:
import time

start = time.perf_counter()

total = 0
for i in range(1_000_000):
    total += i

elapsed = time.perf_counter() - start
print("сумма:", total)
print(f"время: {elapsed:.3f} с")

сумма: 499999500000
время: 0.056 с


## 1.2. Главный вопрос: а если данных станет в 10 раз больше?

Одно измерение не говорит ни о чём. Интересно другое: как время меняется, когда растёт объём данных. Запустим один и тот же код на разных `n`.

In [2]:
def sum_all(n):
    total = 0
    for i in range(n):
        total += i
    return total


for n in [100_000, 1_000_000, 10_000_000]:
    start = time.perf_counter()
    sum_all(n)
    print(f"n = {n:>10}   время = {time.perf_counter() - start:.3f} с")

n =     100000   время = 0.002 с
n =    1000000   время = 0.031 с
n =   10000000   время = 0.284 с


Данных в 10 раз больше - время примерно в 10 раз больше. Такой рост называют **линейным**: во сколько раз выросли данные, во столько же раз вырос расход времени.

## 1.3. А теперь цикл в цикле

Функция ниже для каждого числа пробегает весь список ещё раз. Сравним рост при удвоении `n`.

In [3]:
# O(N * N)
def count_zero_pairs(items):  # len(items) == N
    count = 0
    for a in items: # N
        for b in items: # N
            if a + b == 0:
                count += 1
    return count


for n in [1000, 2000, 4000]:
    data = list(range(-n // 2, n // 2))
    start = time.perf_counter()
    count_zero_pairs(data)
    print(f"n = {n:>5}   время = {time.perf_counter() - start:.3f} с")

n =  1000   время = 0.021 с
n =  2000   время = 0.073 с
n =  4000   время = 0.286 с


Данных в 2 раза больше - время примерно в 4 раза больше. Данных в 4 раза больше - время в 16 раз. Это **квадратичный** рост, и он опасен: на `n = 100 000` такой код будет считать часами.

## 1.4. Секунды - плохая мера, считаем шаги

Секунды зависят от компьютера, загрузки системы и версии Python. У вас числа выше получились не такие, как у меня. Поэтому алгоритмы сравнивают не в секундах, а в **шагах** - сколько раз выполнится тело цикла.

In [ ]:
def steps_linear(n):
    steps = 0
    for i in range(n):
        steps += 1
    return steps


def fun(l1):
    return l1[5]

def steps_quadratic(n):
    steps = 0
    for i in range(n):
        for j in range(n):
            steps += 1
    return steps


for n in [10, 100, 1000]:
    print(f"n = {n:>5}   один цикл: {steps_linear(n):>10}   цикл в цикле: {steps_quadratic(n):>10}")

n =    10   один цикл:         10   цикл в цикле:        100
n =   100   один цикл:        100   цикл в цикле:      10000
n =  1000   один цикл:       1000   цикл в цикле:    1000000


Шаги не зависят от железа: один цикл делает ровно `n` шагов, цикл в цикле - `n * n`. Это и есть то, что мы хотим измерять.

# 2. O-нотация

## 2.1. Короткая запись наблюдения

Вместо «шагов примерно `n`» пишут **O(n)**, вместо «примерно `n * n`» - **O(n^2)**, вместо «число шагов не зависит от `n`» - **O(1)**. Читается «порядка n».

O-нотация отвечает не на вопрос «сколько именно операций», а на вопрос «как растёт время, когда растут данные».

In [ ]:
print(f"{'n':>8} {'O(1)':>8} {'O(n)':>10} {'O(n^2)':>14}")
for n in [10, 100, 1000, 10_000]:
    print(f"{n:>8} {1:>8} {n:>10} {n * n:>14}")

## 2.2. Константы и младшие слагаемые отбрасываем

Код, который делает `3 * n` шагов, и код, который делает `n` шагов, - оба O(n). Разница в 3 раза важна на практике, но не меняет характер роста: удвоили данные - оба замедлились вдвое.

Слагаемые тоже отбрасывают, оставляя самое быстро растущее: `n * n + n` - это O(n^2).

In [ ]:
for n in [10, 100, 1000, 100_000]:
    total = n * n + n
    print(f"n = {n:>7}   n*n + n = {total:>14}   доля слагаемого n = {n / total * 100:.4f}%")

Чем больше `n`, тем меньше значит всё, кроме главного слагаемого. Поэтому его одного и записывают.

## 2.3. Основные классы сложности

| Запись | Как растёт | Пример |
|---|---|---|
| O(1) | не растёт вообще | `items[5]`, `len(items)`, `x in some_set` |
| O(log n) | очень медленно | каждый шаг отбрасывает половину данных |
| O(n) | пропорционально | один проход циклом, `sum`, `max`, `x in items` |
| O(n log n) | чуть хуже, чем пропорционально | `sorted(items)` |
| O(n^2) | катастрофически | цикл в цикле |

Сколько это шагов при `n = 1 000 000`: O(1) - один шаг, O(log n) - около 20, O(n) - миллион, O(n log n) - 20 миллионов, O(n^2) - триллион (это примерно 11 дней на обычном ноутбуке).

In [5]:
# Откуда берётся O(log n): каждый шаг делим данные пополам
n = 1_000_000
steps = 0
while n > 1:
    n //= 2
    steps += 1

print("чтобы поделить миллион пополам до единицы, нужно шагов:", steps)

чтобы поделить миллион пополам до единицы, нужно шагов: 19


## 2.4. Как оценить сложность своего кода

Правила простые:

- всё, что вне циклов, - O(1), это можно не считать;
- один цикл по `n` элементам - O(n);
- цикл внутри цикла - сложности **умножаются**;
- два цикла друг за другом - сложности **складываются**, остаётся большая;
- вызов функции внутри цикла - умножаем на сложность этой функции (это главная ловушка).

In [ ]:
def analyze(items):
    count = 0                       # O(1)

    for x in items:                 # O(1)
        count += 1

    for x in items:                 # O(n)   - складывается с предыдущим: O(n) + O(n) = O(n)
        for y in items:             # O(n)   - вложенный, умножается: O(n^2)
            if x == y:
                count += 1

    return count                    # O(1)


# Итого: O(1) + O(1) + O(1) + O(n^2) = O(n^2) - главный вклад даёт вложенный цикл
print(analyze([1, 2, 3, 2]))

# 3. Пространственная сложность

## 3.1. Что считаем

Пространственная сложность - сколько **дополнительной** памяти заводит код, не считая самих входных данных. Записывается так же:

- O(1) - несколько переменных, сколько бы ни было данных;
- O(n) - новая структура размером со входные данные;
- O(n^2) - например, список всех пар.

Один и тот же результат можно получить за O(1) и за O(n) памяти.

In [ ]:
def sum_positive_list(numbers):
    positive = []                  # O(n) памяти: в худшем случае копия всего списка
    for n in numbers:
        if n > 0:
            positive.append(n)
    return sum(positive)


def sum_positive_counter(numbers):
    total = 0                      # O(1) памяти: одна переменная
    for n in numbers:
        if n > 0:
            total += n
    return total


data = [3, -1, 7, 0, 5]
print(sum_positive_list(data), sum_positive_counter(data))

## 3.2. Измеряем размер объекта

`sys.getsizeof(x)` возвращает размер объекта в байтах. Для списка он показывает размер самого списка (ссылок), без размера элементов, но для сравнения этого достаточно.

In [ ]:
import sys

for n in [0, 10, 1000, 100_000]:
    items = list(range(n))
    print(f"список из {n:>7} элементов: {sys.getsizeof(items):>9} байт")

## 3.3. Лень как экономия памяти

`range`, `enumerate`, `zip` не создают список. Они выдают элементы по одному, поэтому занимают O(1) памяти независимо от `n`. Именно поэтому `range(10 ** 9)` создаётся мгновенно, а `list(range(10 ** 9))` положит компьютер.

In [ ]:
print("range(10**9):        ", sys.getsizeof(range(10 ** 9)), "байт")
print("list(range(100_000)):", sys.getsizeof(list(range(100_000))), "байт")

pairs = zip([1, 2, 3], "abc")
print("zip:                 ", sys.getsizeof(pairs), "байт")

# пока по нему не пройдёшь, элементов в памяти нет
print(list(pairs))

## 3.4. `tracemalloc`: сколько памяти съела функция

`sys.getsizeof` смотрит на один объект. Чтобы увидеть, сколько памяти заняла работа целиком, есть модуль `tracemalloc`: он показывает пиковое потребление между `start()` и `get_traced_memory()`.

In [ ]:
import tracemalloc

def measure(func, data):
    tracemalloc.start()
    func(data)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak


data = list(range(200_000))
print(f"через список:   {measure(sum_positive_list, data):>9} байт")
print(f"через счётчик:  {measure(sum_positive_counter, data):>9} байт")

Результат один и тот же, а памяти первый вариант требует в тысячи раз больше. На больших данных это разница между «работает» и «не хватило памяти».

## 3.5. Время против памяти

Почти всегда ускорение покупается памятью, и наоборот:

- `set(items)` - потратили O(n) памяти, получили поиск за O(1) вместо O(n);
- счётчик вместо списка - сэкономили O(n) памяти, но данные можно пройти только один раз;
- сохранить посчитанное, чтобы не считать заново, - заплатили памятью за время.

Что выбрать, зависит от того, чего не хватает. Если данные маленькие - не выбирайте ничего, пишите просто и понятно.

In [ ]:
names = ["Егор", "Аня", "Пётр", "Лиза"]
banned = ["Пётр", "Олег"]

# O(n * m) времени, O(1) лишней памяти
allowed_slow = [name for name in names if name not in banned]

# O(n + m) времени, O(m) лишней памяти на множество
banned_set = set(banned)
allowed_fast = [name for name in names if name not in banned_set]

print(allowed_slow)
print(allowed_fast)